# Visualizing Sentiment

This notebook takes data from the By the People Comprehensive Transcription Data Package, performs sentiment analysis on the data, and then visualizes the results of the sentiment analysis. The sentiment analysis can be swapped out for any other method that scores documents. The goal of this tutorial is to provide an example of how this data can be analyzed and explored and to provide code that can serve as a starting point for users who would like to create their own projects using the By the People Comprehensive Transcription Data Package.

The notebook includes the following sections:

1. [Fetch the dataset as CSV](#fetch-the-dataset-as-csv)
1. [Clean the transcriptions](#clean-the-transcriptions)
1. [Slice the dataset](#slice-the-dataset)
1. [Analyze sentiment](#analyze-sentiment)
1. [Visualize a single campaign, by asset](#visualize-a-single-campaign-by-asset)
1. [Visualize all campaigns](#visualize-all-campaigns)

## Version

---

Version: 1

Last Run: September 8, 2026 (Python 3.13.5)

---

Author Information:

Created by Library of Congress, Digital Collections Management and Services Division

---

## Prerequisites

The following python libraries can be installed with `pip install`:

- pandas
- nltk
- vaderSentiment
- pillow


In [1]:
import io  # For reading a file from a URL
import zipfile  # For unzipping the CSV data file

import nltk  # For sentiment analysis
import pandas as pd  # For data manipulation
import requests  # For making http requests
from IPython.display import HTML  # For displaying visualizations in this notebook
from nltk.sentiment.vader import SentimentIntensityAnalyzer  # For sentiment analysis
from PIL import ImageColor  # For color manipulation (Pillow)


We’ll also need to download the [Natural Language Toolkit (NLTK)](https://www.nltk.org/) Vader lexicon.


In [2]:
nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\rtrent\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

## Fetch the dataset as CSV

This cell will fetch the CSV ZIP, unzip it, and read it into a `pandas` dataframe named `data`.


In [3]:
zipped_csv_url = "https://tile.loc.gov/storage-services/master/gdc/gdcdatasets/2026395156/2026395156_csv.zip"

# Try to fetch the CSV file
try:
    response = requests.get(zipped_csv_url, stream=True)
    response.raise_for_status()
    print(f"CSV fetched!")
except Exception as e:
    print(
        f"Oops! There was an issue downloading the ZIP. Try re-running this cell. Here's the error message: {e}"
    )
    raise

# Try to unzip and read the CSV into a pandas dataframe
try:
    # Unzip
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        csv_filename = z.namelist()[0]
        with z.open(csv_filename) as f:
            # Read into a pandas dataframe
            data = pd.read_csv(f, dtype=str)
    print('Done! The dataset is read into a pandas dataframe named "data".')
except Exception as e:
    print(
        f"Oops! There was a problem opening the CSV into a pandas dataframe. Here's the error message: {e}"
    )

CSV fetched!
Done! The dataset is read into a pandas dataframe named "data".


We can take a peak at the first two rows of the dataframe. There are many columns, so not all columns will show. `NaN` indicates an empty dataframe cell.


In [4]:
data.head(2)

,Campaign,Project,Item,ItemId,Asset,AssetId,AssetStatus,DownloadUrl,Transcription,Tags,...,JP2,SegmentUrl,ResourceUrl,SubjectHeadings,ContributorNames,OriginalFormat,Medium,CallNumber,Repository,Notes
0,"Seers, Spiritualists, and the Spirit World: Th...",The crystal: a record of visions and conferenc...,The crystal : a record of visions and conferen...,2010414646,2010414646-1,508113,completed,https://tile.loc.gov/image-services/iiif/servi...,NaN,Cover; Cover; leather; Cover; leather; Cover; ...,...,https://tile.loc.gov/storage-services/service/...,https://www.loc.gov/resource/rbc0001.2018houd1...,https://www.loc.gov/resource/rbc0001.2018houd1...,"['Hockley, Frederick,--1808-1885--Notebooks, s...","['Hockley, Frederick, 1808-1885.', 'Harry Houd...",['manuscript/mixed material'],"['11 v. (v. 4-5, 7-15) : ill. (some col.) ; 23...",['BF1335 .H63 1853'],[],NaN
1,"Seers, Spiritualists, and the Spirit World: Th...",The crystal: a record of visions and conferenc...,The crystal : a record of visions and conferen...,2010414646,2010414646-2,508114,completed,https://tile.loc.gov/image-services/iiif/servi...,147\r\n\r\nEx Libris\r\nHoudini\r\nTHE LIBRARY...,fabric; Harry Houdini; bequest; seal; Harry Ho...,...,https://tile.loc.gov/storage-services/service/...,https://www.loc.gov/resource/rbc0001.2018houd1...,https://www.loc.gov/resource/rbc0001.2018houd1...,"['Hockley, Frederick,--1808-1885--Notebooks, s...","['Hockley, Frederick, 1808-1885.', 'Harry Houd...",['manuscript/mixed material'],"['11 v. (v. 4-5, 7-15) : ill. (some col.) ; 23...",['BF1335 .H63 1853'],[],NaN


## Clean the transcriptions

We need to clean the transcriptions up a little for analysis. We will remove newlines, which appear as "\r\n" and replace that with a blank space.

Our transcription data is entered as free text and is therefore uncontrolled. You can apply additional data cleaning and standardization at this stage. Information about how text is transcribed can be found on the [By the People website](https://crowd.loc.gov/get-started/how-to-transcribe/).


In [5]:
data["clean_trnscrpt"] = data["Transcription"].str.replace("\r\n", " ", regex=False)

## Slice the dataset

We are working with a large corpus of data, so we will first test our processes on a slice of the dataset. We can use pandas to filter the dataset to just one By the People campaign. You can edit the first code cell here to analyze and visualize a different campaign. A full list of campaigns from the dataset can be found on the Data at a Glance page.


✍🏽 **Try editing this cell!! ⤵︎**


In [6]:
campaign_filter = "Rosa Parks: In Her Own Words"

In [7]:
filtered_data = data[data["Campaign"] == campaign_filter].copy()

print(
    'Done! A new dataframe "filtered_data" contains data from the '
    f'"{campaign_filter}" transcription campaign.'
)
print(
    f"This dataframe includes {len(filtered_data)} rows, each describing "
    "one document from that campaign."
)

Done! A new dataframe "filtered_data" contains data from the "Rosa Parks: In Her Own Words" transcription campaign.
This dataframe includes 1769 rows, each describing one document from that campaign.


## Analyze sentiment

For this tutorial, we are using the Natural Language Processing (NLP) technique of sentiment analysis. Our code employs NLTK VADER (Valence Aware Dictionary and sEntiment Reasoner) lexicon, which contains a dictionary of words with an assigned status of positive, negative, or neutral. In this code section, the words in each asset are reviewed, and then the asset is assigned a numerical score between -1 and 1. These scores will be used in the visualization later on.

In this tutorial, we are using one technique and one tool to explore our data. Sentiment analysis results may vary based on the tool used and the corpus of data. The results in this tutorial are intended to demonstrate one method for reviewing and visualizing data, not to serve as a definitive analysis of By the People transcription data.

You can update the `generate_scores()` function in the code below to use a different method of producing a "score" within a number range to each asset. If you do so, make sure you update the `max_score` and `min_score` values to match the scores produced by your method.


✍🏽 **Try editing this cell !!** (⚠️Advanced) **⤵︎**


In [8]:
max_score = 1
min_score = -1


def generate_scores(text):
    """
    This function takes a block of trascription text, and outputs a score
    between -1 and 1.
    In this example, the score represents the overall "sentiment" or tone
    of the text, as characterized by the 2014 VADER sentiment analysis tool
    in Natural Language Toolkit (NLTK).
    """
    nltk_analyzer = SentimentIntensityAnalyzer()
    sentiment_dict = nltk_analyzer.polarity_scores(text)
    return sentiment_dict["compound"]

Before applying our `generate_scores()` function to our dataframe, we'll want to drop any rows (pages) that don't have text.


In [9]:
data_with_txt = filtered_data[filtered_data["clean_trnscrpt"].notnull()].copy()

Now we're ready to apply the `generate_scores()` function!
Note: This cell may run for a long time if you updated the code above to filter to a larger campaign. We pre-populated with the Rosa Parks collection, which is a small one.


In [10]:
data_with_txt["sentiment_score"] = data_with_txt["clean_trnscrpt"].apply(
    generate_scores
)

Let's look at the results.

The table below shows the scores generated for a sample of assets. According to VADER’s classifications, scores closer to -1 are more "negative" and scores closer to 1 are more "positive." Scores of 0 are "neutral."


In [11]:
print("Table: Preview of resulting scores:")

data_with_txt[["sentiment_score", "clean_trnscrpt"]][0:10]

Table: Preview of resulting scores:


,sentiment_score,clean_trnscrpt
148812,0.0000,ROSA PARKS FAMILY PAPERS Letters to ...
148813,-0.8765,"May 15, 1950. Patterson Calif My Dear Daughte..."
148814,0.0000,Jas. McCauley ...
148815,0.9490,for these many years. in Grose desertion of a ...
148816,0.0000,ROSA PARKS FAMILY PAPERS Lette...
148817,0.3818,"Pine Level, Ala Sept. 16, 1936 Dear Mother"
148818,0.9842,"July 24, '46 Detroit, Mich. My dear daughter,..."
148819,0.9810,I came back here Saturday. Aunt Sophronia and ...
148821,0.0000,"5 Eliot St it 906 etroit, 1, Mich. DETROIT, M..."
148822,0.9030,"Sept. 9, 1946 My dear daughter,--How are you ..."


By default, `pandas` doesn't show us much of each column. We can increase the number of characters that display in a column to better view the transcription data.


In [12]:
pd.options.display.max_colwidth = 500

In [13]:
print("Table: Preview of resulting scores, with wider columns")

data_with_txt[["sentiment_score", "clean_trnscrpt"]][0:10]

Table: Preview of resulting scores, with wider columns


,sentiment_score,clean_trnscrpt
148812,0.0000,"ROSA PARKS FAMILY PAPERS Letters to and from Rosa Parks 1950 Box 2 Folder 1 McCauley, James (father)"
148813,-0.8765,"May 15, 1950. Patterson Calif My Dear Daughter I Received Your letter of may 5th. a few days a-go and was Indeed glad to hear from you. and the others that go to make up the family. and to learn that all was well. and that Sylvester was married and doing well. was indeed a pleasure. Yes I wrote to sister Jessee Bell. at Eufaula Ala first. she informed me of Deaths of Mother Bro Robert and George. I also written to sister Addie at Ozark. and inquired of you. she said you was still in M..."
148814,0.0000,Jas. McCauley MODESTO [postmark] G. D. Modesto Calif MAY 15 1 30 pm 1950 CALIF. Mrs. Rosa L. Parks 326 Columbia Ave. Montgomery Alabama
148815,0.9490,"for these many years. in Grose desertion of a a good Wife and two of the sweetest children every lived. and at a time when I was needed most. I dread to mention these unpleasant events to you. but as I am now on the sun set side of life. I need to unbrdon myself of these Sins of such Crual nature. I some time think they are unforgiveable. Please you and Mother forgive me. I will Write Brother soon. I and glad to say tat t hes writing that I am well and are in very good health, I d..."
148816,0.0000,"ROSA PARKS FAMILY PAPERS Letters to and from Rosa Parks 1936, 1946 Box 2 Folder 2 McCauley, Leona (mother)"
148817,0.3818,"Pine Level, Ala Sept. 16, 1936 Dear Mother"
148818,0.9842,"July 24, '46 Detroit, Mich. My dear daughter,---How are you and family to day? fine I hope. I am all OK. Also the Alexanders am still enjoying it here & wish you could be here too. I went out to Thomas' home last Tuesday week and stayed untill Saturday. They have a nice place out there and I like it. Lucille took me to the show one night. and to visit some of her friends. [while] I enjoyed my stay very much. All the children have grown a lots. Caroline is just like Fran was when she was ..."
148819,0.9810,"I came back here Saturday. Aunt Sophronia and I went to church Sunday as usual, The Bishop preached, the service was fine. Monday night I was invited to the show by Shellie Stenson's wife. She is a very pretty young woman with a very pleasing personallity. I am enclosing a pamplet of the real life drama of Deep are the roots. I wish you could have seen it. It was really good and worth seeing. I think you read the book of course, but it is so real in the play. Much better than the movie, an..."
148821,0.0000,"5 Eliot St it 906 etroit, 1, Mich. DETROIT, MICH. 6 JUL 25 3 PM 1946 Mrs Rosa L. Parks 22 Mill St. Cleveland Ct. Montgomery, 5, Ala."
148822,0.9030,"Sept. 9, 1946 My dear daughter,--How are you and Parks getting along? Fine I hope. Brother arrived safely Friday morning, in fine shape except a little sleepy, tired and hungry. So he ate, relaxed and took a nap. We went round to Roxanna's for a short while. Thomas and Bea McDaniel came over that night and he went home with them. He called me this morning, said he would be back over this after noon. He is looking for work this morning. He and little William McWhirter is to-gether. Luc..."


In the above table, we can see that transcriptions containing addresses or archival box and folder information are given neutral scores. The one transcription with a negative score includes a reference to the death of two people. Of the remaining transcriptions with more positive scores, the text of the fourth row seems to depict more negative sentiments than are reflected in the assigned sentiment score. From these examples, we can infer that the results of our sentiment analysis may vary in their accuracy. As previously stated, this notebook is demonstrating just one way to review and visualize By the People transcription data. It is not intended to serve as a definitive analysis of the data.


## Visualize a single campaign, by asset


The following code will visualize the sentiment scores assigned above using a color spectrum. You can edit the cell below with new hexadecimal codes to change the colors. The codes can be generated using an online color picker. This tutorial has selected red as the maximum color for positive sentiments, blue as the minimum color for negative sentiments, and white as the middle color for neutral sentiments.

✍🏽 **Try editing this cell!! ⤵︎**


In [14]:
max_color = "#e01919"  # Positive sentiment
middle_color = "#ffffff"  # Neutral sentiment
min_color = "#0c13c7"  # Negative sentiment

We use Pillow (which `ImageColor` is imported from) to convert the hex colors above into RGB tuples like `(235, 52, 180)`.


In [15]:
max_color_rgb = ImageColor.getcolor(max_color, "RGB")  # e.g., (112, 237, 9)
middle_color_rgb = ImageColor.getcolor(middle_color, "RGB")  # e.g., (245, 245, 245)
min_color_rgb = ImageColor.getcolor(min_color, "RGB")  # e.g., (235, 52, 180)

This cell creates a function to convert our scores to be within the 0-1 range, which will allow us to turn them into percentage decimals. If you updated the code to replace sentiment analysis with a different type of analysis, you may also need to update the `min_score` and `max_score` values here.


In [16]:
def normalize_score(score, min_score=min_score, max_score=max_score):
    """
    Converts scores into the 0 to 1 range (i.e., converts into
    percentages)
    """
    score_range = max_score - min_score
    normalized_score = score_range - (max_score - score)
    score_perc = normalized_score / score_range
    return score_perc

Let’s apply our function. We are working in a `pandas` dataframe, so we can use a simple bit of code that applies our function to the `sentiment_score` column, and create a new `sentiment_score_norm` column to hold the outputs.


In [17]:
data_with_txt["sentiment_score_norm"] = data_with_txt["sentiment_score"].apply(
    normalize_score
)

Let's look at the results! We have a lot of columns, so we'll just look at three columns
To make the outputs more manageable, we'll change the pandas setting again to show fewer max characters


In [18]:
pd.options.display.max_colwidth = 100
print("Table: Preview results with the new 'sentiment_score_norm' column.")
data_with_txt[["sentiment_score_norm", "sentiment_score", "Transcription"]]

Table: Preview results with the new 'sentiment_score_norm' column.


,sentiment_score_norm,sentiment_score,Transcription
148812,0.50000,0.0000,ROSA PARKS FAMILY PAPERS Letters to and from Rosa Parks 1950 \r\nBox 2 Folder...
148813,0.06175,-0.8765,"May 15, 1950.\r\nPatterson Calif\r\n\r\nMy Dear Daughter\r\nI Received Your letter of\r\nmay 5th..."
148814,0.50000,0.0000,Jas. McCauley MODESTO [postmark]\r\nG. D. ...
148815,0.97450,0.9490,for these many years. in Grose\r\ndesertion of a a good Wife and \r\ntwo of the sweetest childre...
148816,0.50000,0.0000,"ROSA PARKS FAMILY PAPERS Letters to and from Rosa Parks 1936, 1946..."
...,...,...,...
150575,0.75465,0.5093,MRA. ROSA\r\nPARKS—\r\nWELCOME \r\nTO OUR\r\nCOMMUNITY!\r\nWOOSTER HIGH\r\nBLACK\r\nSTUDENT\r\nU...
150576,0.99815,0.9963,"PERSONAL NOTES OF WELCOME FROM OUR MEMBERS \r\n\r\nDear Mrs. Parks,\r\nI have read of your stren..."
150577,0.99625,0.9925,"wrong in society like you did. ""Thank you"" for all you have\r\ndone. You are my idol. \r\nYour F..."
150578,0.94895,0.8979,Mrs. Rosa Parks I would like to thank \r\nyou for stopping buy and taking \r\ntime to talk to u...


Now we need a function to assign each normalized sentiment score a color. The color will lie along the gradient created by our min, middle, and max colors.


In [19]:
def blend_colors(
    p,
    max_color_rgb=max_color_rgb,
    middle_color_rgb=middle_color_rgb,
    min_color_rgb=min_color_rgb,
):
    """
    Calculates the color at p percent between a max, middle, and min color.

    Colors should be input in the format of tuples like (R, G, B) where
    R,G, and B are integers between 0 and 255

    p is a decimal between 0 and 1 (percentage)
    """
    if p > 0.5:
        return tuple(
            int(middle_color_rgb[i] + (max_color_rgb[i] - middle_color_rgb[i]) * p)
            for i in range(3)
        )
    elif p < 0.5:
        return tuple(
            int(middle_color_rgb[i] + (min_color_rgb[i] - middle_color_rgb[i]) * p)
            for i in range(3)
        )
    elif p == 0.5:
        return middle_color_rgb

Ok, let's apply this to our dataframe to create another column, `color`, which will hold the RGB values representing the sentiment score of each transcription


In [20]:
data_with_txt["color"] = data_with_txt["sentiment_score_norm"].apply(blend_colors)

And let's take a look at results, picking just the relevant columns to show again


In [21]:
data_with_txt[["color", "sentiment_score_norm", "sentiment_score", "Transcription"]]

,color,sentiment_score_norm,sentiment_score,Transcription
148812,"(255, 255, 255)",0.50000,0.0000,ROSA PARKS FAMILY PAPERS Letters to and from Rosa Parks 1950 \r\nBox 2 Folder...
148813,"(239, 240, 251)",0.06175,-0.8765,"May 15, 1950.\r\nPatterson Calif\r\n\r\nMy Dear Daughter\r\nI Received Your letter of\r\nmay 5th..."
148814,"(255, 255, 255)",0.50000,0.0000,Jas. McCauley MODESTO [postmark]\r\nG. D. ...
148815,"(224, 30, 30)",0.97450,0.9490,for these many years. in Grose\r\ndesertion of a a good Wife and \r\ntwo of the sweetest childre...
148816,"(255, 255, 255)",0.50000,0.0000,"ROSA PARKS FAMILY PAPERS Letters to and from Rosa Parks 1936, 1946..."
...,...,...,...,...
150575,"(231, 81, 81)",0.75465,0.5093,MRA. ROSA\r\nPARKS—\r\nWELCOME \r\nTO OUR\r\nCOMMUNITY!\r\nWOOSTER HIGH\r\nBLACK\r\nSTUDENT\r\nU...
150576,"(224, 25, 25)",0.99815,0.9963,"PERSONAL NOTES OF WELCOME FROM OUR MEMBERS \r\n\r\nDear Mrs. Parks,\r\nI have read of your stren..."
150577,"(224, 25, 25)",0.99625,0.9925,"wrong in society like you did. ""Thank you"" for all you have\r\ndone. You are my idol. \r\nYour F..."
150578,"(225, 36, 36)",0.94895,0.8979,Mrs. Rosa Parks I would like to thank \r\nyou for stopping buy and taking \r\ntime to talk to u...


Great, we have our colors assigned! Now we'll make a function to display them in a pretty way in this notebook.

The function by default is configured to show up to 1,000 squares. You can update the max number in the code block below to display more squares.


In [22]:
def show_color_squares(
    colors,
    size=20,
    max=1000,
):
    """
    Function to display colors in this notebook.

    colors - List of RGB color values like (10,10,10)
    size - Pixel size of squares to display
    max - Max number of squares to display
    """
    squares = "".join(
        f"<span style='width:{size}px;height:{size}px;"
        f"display:inline-block;margin:2px;"
        f"background:rgb{c};'"
        f"></span>"
        for c in colors[:max]
    )
    return HTML(f"<div>{squares}</div>")

In [23]:
print(f"Sentiment of pages in the {campaign_filter} campaign, by color:")
colors = data_with_txt["color"].tolist()
show_color_squares(colors)

Sentiment of pages in the Rosa Parks: In Her Own Words campaign, by color:


In [24]:
print(f"Average sentiment color for all pages in the {campaign_filter} campaign:")
show_color_squares([blend_colors(data_with_txt["sentiment_score_norm"].mean())])

Average sentiment color for all pages in the Rosa Parks: In Her Own Words campaign:


## Visualize all campaigns

The final section creates a single sentiment score for each campaign and compares campaigns by color. This section takes a while to run, because NLTK has to create sentiment scores for a large amount of text. To mitigate this, I've set up the code below to sample up to 100 items from each campaign. You (Madeline) might want to play around with this a little to see if it's worth increasing that number. Also good to give users the option to play around with it as well

We'll return to the dataframe we first created, `data`, before we filtered to a single campaign, cleaned up the dataset, and calculated sentiment scores for one of the campaigns. This dataframe has the entire original dataset.

The first thing we want to do is a bit complicated. We want to collapse our dataframe so that there's just one row per campaign, and we combine all the transcriptions for that campaign. We'll use some fancy `pandas` syntax for this.


Users can set a maximum sample here. This is the max number of rows per campaign that will be used to calculate the campaign's overall sentiment

✍🏽 **Try editing this cell!! ⤵︎**


In [25]:
max_sample = 100

In [26]:
# Drop blank pages without transcriptions
data_with_transcriptions = data[data["Transcription"].notnull()].copy()

This next cell does something tricky, that isn't easy to do with simple code. It takes a sample of up to `max_sample` rows per campaign. If the user changes `max_sample` to 1,000 but a campaign only has 500 rows, then it just takes all 500. There isn't a pretty way to do this, so the code is a little complex.


In [27]:
def agg_sample(data, n):
    """
    This is the function we'll use to sample *up to* the max_sample number of rows
    per campaign. This function allows you to use a large max_sample number without
    worrying whether there are campaigns with fewer rows.
    """
    sample_size = min(n, len(data))
    return data.sample(n=sample_size, random_state=99)


# Now we'll use a special Pandas function called "lambda" to apply our function
sampled_campaigns = (
    data_with_transcriptions
    .groupby("Campaign")
    .apply(lambda x: agg_sample(x, n=max_sample), include_groups=False)
    .reset_index()
)

Now we'll repeat our data cleaning and sentiment-score-generating steps that we did before.

This cell may take a while, if the user has increased the max_sample above 100


In [28]:
# Generate the raw sentiment scores
sampled_campaigns["sentiment_score"] = sampled_campaigns["clean_trnscrpt"].apply(
    generate_scores
)

# Normalize the sentiment scores 0 - 1
sampled_campaigns["sentiment_score_norm"] = sampled_campaigns["sentiment_score"].apply(
    normalize_score
)

Let's get the average (mean) sentiment score for each campaign


In [29]:
sentiment_by_campaign = (
    sampled_campaigns
    .groupby("Campaign")["sentiment_score_norm"]
    .mean()
    .to_frame()
    .reset_index()
)
sentiment_by_campaign

,Campaign,sentiment_score_norm
0,"""Mr. President: What Will You Do for Woman Suffrage?"": Woodrow Wilson’s File 89",0.814287
1,"""Such Eventful Times"": Women and the American Civil War",0.719313
2,"""This Hell-upon-earth of a Prison"": Samuel J. Gibson's Andersonville Diary",0.328124
3,African American Perspectives In Print,0.655704
4,American Creativity: Early Copyright Title Pages,0.568993
5,Anna E. Dickinson Papers,0.762514
6,Artistic Trio: Georgia O’Keeffe & Alfred Stieglitz letters to Henwar Rodakiewicz,0.699027
7,At the Library and in the Field: John and Alan Lomax Papers,0.767161
8,Branch Rickey: Changing the Game,0.745932
9,Brothers in Arms: The Gladstone Afro-American Military Collection,0.597348


As we did before, let's translate those sentiment scores into colors.


In [30]:
sentiment_by_campaign["color"] = sentiment_by_campaign["sentiment_score_norm"].apply(
    blend_colors
)
sentiment_by_campaign

,Campaign,sentiment_score_norm,color
0,"""Mr. President: What Will You Do for Woman Suffrage?"": Woodrow Wilson’s File 89",0.814287,"(229, 67, 67)"
1,"""Such Eventful Times"": Women and the American Civil War",0.719313,"(232, 89, 89)"
2,"""This Hell-upon-earth of a Prison"": Samuel J. Gibson's Andersonville Diary",0.328124,"(175, 177, 236)"
3,African American Perspectives In Print,0.655704,"(234, 104, 104)"
4,American Creativity: Early Copyright Title Pages,0.568993,"(237, 124, 124)"
5,Anna E. Dickinson Papers,0.762514,"(231, 79, 79)"
6,Artistic Trio: Georgia O’Keeffe & Alfred Stieglitz letters to Henwar Rodakiewicz,0.699027,"(233, 94, 94)"
7,At the Library and in the Field: John and Alan Lomax Papers,0.767161,"(231, 78, 78)"
8,Branch Rickey: Changing the Game,0.745932,"(231, 83, 83)"
9,Brothers in Arms: The Gladstone Afro-American Military Collection,0.597348,"(236, 117, 117)"


Let's sort from negative to positive sentiment


In [31]:
sentiment_by_campaign.sort_values("sentiment_score_norm", inplace=True)

Now, let's visualize. We'll make a slightly different function than before, in order to label each square


In [32]:
def show_labelled_color_squares(colors, labels, size=20):
    """
    Function to display colors with Campaign labels next to them

    colors - List of RGB color values like (10,10,10)
    labels - List of strings, to display as labels next to the colors.
        Should be the same length as the colors list.
    size - Pixel size of squares to display
    """
    squares = "".join(
        f"<span style='width:{size}px;height:{size}px;"
        f"display:inline-block;margin:2px 10px 2px 2px;"
        f"background:rgb{c};'></span><span>{labels[i]}</span><br>"
        for i, c in enumerate(colors)
    )
    return HTML(f"<div>{squares}</div>")

In [33]:
print(f"Overall sentiment of campaigns, by color:")
campaign_colors = sentiment_by_campaign["color"].tolist()
campaign_labels = sentiment_by_campaign["Campaign"].tolist()
show_labelled_color_squares(campaign_colors, campaign_labels)

Overall sentiment of campaigns, by color:
